# iQuHack state-preparation workshop

Run the reference exercises locally. Verify coherent arithmetic, fixed-point addition, indexed amplitude loading, and exponential preparation followed by QFT.

## See which stages use high-level synthesis

The editable arithmetic qfunc below requests coherent output computation from
two three-bit inputs. The other native helpers perform bounded probability
loading, indexed indicator rotations and preparation followed by QFT. The signed
fixed-point qfunc also executes at the original fourteen-bit budget, with
its explicit Circuit adder retained as the independent coherent reference.

Try changing the arithmetic offset and enumerate the new output range before
synthesis. For loading, probabilities and indicator amplitudes have different
normalization rules. The indexed amplitude experiment retains 2048 draws and
its full-state reference; sampled frequency alone cannot certify complex phase
or clean workspace. Compare the signed program's width-constrained Fourier route
with its fewer-instruction default ripple route.


Use the installed local FlagQuantum runtime.

In [ ]:
import examples.synthesis.community.hackathons.iquhack_2025.workshop.overview.main as report
import examples.synthesis.community.hackathons.iquhack_2025.workshop.overview.workflow as workflow
import torch
import flagquantum as fq
from flagquantum import synth
from typing import Annotated
from flagquantum.synth import Output, QNum, allocate, hadamard_transform, qfunc


@qfunc
def arithmetic(
    x: Output[Annotated[QNum, 3]],
    y: Output[Annotated[QNum, 6]],
    z: Output[Annotated[QNum, 3]],
) -> None:
    allocate(x)
    allocate(z)
    hadamard_transform(x)
    hadamard_transform(z)
    y |= 2 * x + 4 * z + 2


preview = synth.synthesize(arithmetic, max_qubits=14, max_workspace=2)
expected = torch.zeros(4096, dtype=torch.complex128)
for x in range(8):
    for z in range(8):
        expected[(x << 9) | ((2 * x + 4 * z + 2) << 3) | z] = 1 / 8
preview_state = fq.run(preview).statevector().reshape(4096, -1)
torch.testing.assert_close(preview_state[:, 0], expected, atol=3e-12, rtol=0)
assert float(preview_state[:, 1:].abs().square().sum()) < 1e-22
print("Editable arithmetic resources:", preview.report["resources"])


Run the exercises locally with a seeded sampling generator.

In [ ]:
assert workflow.SEED == 5


Use the provided native reference solution for both uniform input registers.

In [ ]:
arithmetic = workflow.arithmetic()


Inspect the synthesized coherent arithmetic and its complete-state check.

In [ ]:
print("Coherent unsigned arithmetic:", arithmetic)


Regenerate the original NumPy seed-zero probability array.

In [ ]:
probabilities = workflow.random_probabilities()
print(probabilities)


Synthesize the probability preparation at the original L2 threshold.

In [ ]:
prepared, distance = workflow.prepare_random(probabilities)
print("Seeded 32-state preparation:", prepared)
print("Preparation probability L2 error:", distance)


Add the original signed fixed-point operands in half-units.

The native declaration below requests the original 14-qubit budget and zero extra work.
FlagQuantum borrows and restores an input word internally to synthesize the XOR sum.
Named numeric reading returns signed half-unit values without bitstring decoding.
This width-constrained route uses 89 logical instructions, including preparation;
the default ripple route uses 20 qubits and 47 instructions. The original 14-qubit
Circuit remains the independent coherent reference. No gate-speed advantage is claimed.


In [ ]:
values, signed = workflow.fixed_point()


@qfunc
def signed_addition(
    x: Output[QNum[3]],
    y: Output[QNum[5, True, 1]],
    result: Output[QNum[6, True, 1]],
) -> None:
    x |= 5
    y |= -7.5
    result |= x + y


native_signed = synth.synthesize(
    signed_addition,
    max_qubits=14,
    max_workspace=0,
    max_operations=1000,
    memory_limit_bytes=1024**2,
)
native_signed_result = fq.run(native_signed)
print(dict(native_signed.numeric_probabilities(
    native_signed_result, registers=("x", "y", "result")
)))
print("Native signed execution resources:", native_signed.report["resources"])


Inspect the 14-bit circuit and all 256 coherent input pairs.

In [ ]:
print("Signed fixed-point operands and sum:", values)
print("Signed fixed-point all-branch check:", signed)


Load every one of the 32 fractional squares into the indicator.

In [ ]:
assert len(workflow.AMPLITUDES) == 32


Draw the original 2,048 joint samples.

In [ ]:
amplitude, sample = workflow.amplitude_assignment()


Read the joint index and indicator counts.

In [ ]:
assert len(sample.joint) >= 32


Read the index marginal from the same observed draws.

In [ ]:
report.print_amplitude_counts(sample)


Inspect the synthesized indexed-rotation Circuit resources.

In [ ]:
print("Fractional square amplitude loading:", amplitude)


Use a five-qubit exponentially decaying probability grid and apply the QFT.

In [ ]:
exponential, distribution, outcome = workflow.exponential_qft()


Check the Fourier output and perform the one-shot bonus.

In [ ]:
print("Exponential QFT state check:", exponential)
print("Exponential QFT probability at zero:", distribution[0])


Report the actual sampled result from the bonus run.

In [ ]:
print("Exponential QFT bonus draw:", outcome)
